# Phase 1: Auditable Incident Deduplication

This notebook is the analyst-facing view of the Phase 1 v3 pipeline. It preserves every source report and assigns each configured same-crossing comparison pair exactly one deterministic decision: `auto_merge` or `keep_distinct`.

Duration categories are documented proxy intervals, not measured endpoints. `Date/Time` is interpreted as UTC under the approved source contract. Crossing-local timestamps are derived from the current Form 71 coordinate and historical IANA daylight-saving rules; neither field establishes physical-event truth.

In [ ]:
import hashlib
import importlib
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Workflow Configuration Switches
# -------------------------------------------------------------------------
REUSE_STEP_5_CHECKPOINT = False
RUN_REPEATABILITY_CHECK = False

# Repeatability mode takes precedence and disables checkpoint reuse
if RUN_REPEATABILITY_CHECK:
    REUSE_STEP_5_CHECKPOINT = False

repo_root = Path(r"C:/Projects/Blocked-Crossing-Prediction")
analysis_dir = repo_root / "analysis"
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import incident_deduplication
incident_deduplication = importlib.reload(incident_deduplication)
run_phase_1 = incident_deduplication.run_phase_1

authoritative_path = repo_root / "data" / "blocked_crossings_2020through2025.xlsx"
reconciliation_path = repo_root / "data" / "blocked_crossings_2025.xlsx"
inventory_path = repo_root / "data" / "Crossing_Inventory_Data_(Form_71)_-_Current_20260707.csv"
config_path = analysis_dir / "incident_deduplication_config.json"
output_dir = repo_root / "analysis_outputs" / "deduplication" / "v3"

# Record initial SHA-256 hashes strictly for raw input integrity verification
def compute_file_hash(filepath: Path) -> str:
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
    return hasher.hexdigest()

initial_input_hashes = {
    "authoritative": compute_file_hash(authoritative_path),
    "reconciliation": compute_file_hash(reconciliation_path),
    "inventory": compute_file_hash(inventory_path),
}

# -------------------------------------------------------------------------
# Pipeline Execution Logic
# -------------------------------------------------------------------------
if RUN_REPEATABILITY_CHECK:
    print("Running Repeatability Diagnostic Mode (Two Fresh Runs)...")
    
    # Run 1
    dir_run1 = output_dir / "repeatability_run1"
    result_run1 = run_phase_1(
        authoritative_path,
        reconciliation_path,
        inventory_path,
        config_path,
        dir_run1,
        reuse_step_5_checkpoint=False,
    )
    
    # Run 2
    dir_run2 = output_dir / "repeatability_run2"
    result_run2 = run_phase_1(
        authoritative_path,
        reconciliation_path,
        inventory_path,
        config_path,
        dir_run2,
        reuse_step_5_checkpoint=False,
    )
    
    # Primary result set for notebook display
    result = result_run1
    
    # Compare summary metrics between independent runs
    two_real_data_runs_match = result_run1.summary == result_run2.summary
    result.summary["repeatability_check"] = "passed" if two_real_data_runs_match else "failed"

else:
    # Single run mode (Fresh execution or Resumed from Step 5 Checkpoint)
    result = run_phase_1(
        authoritative_path,
        reconciliation_path,
        inventory_path,
        config_path,
        output_dir,
        reuse_step_5_checkpoint=REUSE_STEP_5_CHECKPOINT,
    )
    result.summary["repeatability_check"] = "not_run"

# Prove raw inputs were not mutated during pipeline execution
post_input_hashes = {
    "authoritative": compute_file_hash(authoritative_path),
    "reconciliation": compute_file_hash(reconciliation_path),
    "inventory": compute_file_hash(inventory_path),
}
assert initial_input_hashes == post_input_hashes, "Raw input files were modified during execution!"

# Output Summary and Immediate Validations
print(json.dumps(result.summary, indent=2))

assert result.validations["source_rows_map_once"]
assert result.validations["only_configured_auto_merge_tiers"]

if RUN_REPEATABILITY_CHECK:
    assert result.summary.get("repeatability_check") == "passed", "Repeatability check failed between the two runs."

## Inventory, normalization, and provenance

Unknown duration values remain unmapped; invalid crossing IDs and timestamps remain in the source table and receive documented exceptions rather than canonical incidents.

In [ ]:
inventory_profile = json.loads((output_dir / 'inventory_profile.json').read_text(encoding='utf-8'))
pd.DataFrame([inventory_profile['duration_normalization'], inventory_profile['crossing_id_status']], index=['duration status', 'crossing ID status']).T.fillna(0)

## UTC and crossing-local time

UTC remains the canonical comparison timestamp. Rows without a coordinate-derived IANA zone remain UTC-only and are flagged; no state-level fallback is used.

In [ ]:
timezone_coverage = pd.read_csv(output_dir / 'timezone_assignment_diagnostics.csv')
local_time_diagnostics = pd.read_csv(output_dir / 'local_time_diagnostics.csv')
display(timezone_coverage)
local_time_diagnostics.sort_values(['iana_time_zone', 'reported_local_hour']).head(30)

In [ ]:
source_reports = pd.read_parquet(output_dir / 'source_reports_with_ids.parquet')
source_reports.loc[source_reports['timezone_assignment_status'].eq('assigned'), [
    'source_row_id', 'norm_crossing_id', 'reported_at_utc', 'reported_at_local',
    'iana_time_zone', 'utc_offset_minutes', 'State', 'City'
]].head(20)

## Deterministic pair decisions and uncertainty

Every configured comparison pair is decided as `auto_merge` or `keep_distinct`. `decision_basis` records why, while `uncertainty_flag` and `uncertainty_basis` describe evidence strength without creating an unresolved workflow state. A `possible_temporal_overlap` flag is proxy evidence, not proof that two reports describe the same physical event.

In [ ]:
deduplication_summary = json.loads((output_dir / 'deduplication_summary.json').read_text(encoding='utf-8'))
display(pd.Series(deduplication_summary))
pair_decision_summary = json.loads((output_dir / 'pair_decision_summary.json').read_text(encoding='utf-8'))
display(pd.Series(pair_decision_summary))
pair_decisions = pd.read_parquet(output_dir / 'pair_decisions.parquet')
decision_audit_sample = pd.read_csv(output_dir / 'pair_decision_audit_sample.csv')
decision_audit_sample.head(20)

## Reconciliation, diagnostics, and gate

The 2025 comparison is a normalized full-row multiset comparison, including duplicate multiplicity. A no-report interval must not be called unblocked.

In [ ]:
reconciliation_summary = json.loads((output_dir / 'reconciliation_summary.json').read_text(encoding='utf-8'))
gate_report = json.loads((output_dir / 'phase_1_gate_report.json').read_text(encoding='utf-8'))
display(pd.Series(reconciliation_summary))
display(pd.read_csv(output_dir / 'timestamp_granularity_by_year.csv'))
display(pd.read_csv(output_dir / 'diagnostics_by_year.csv'))
gate_report